In [1]:
import pandas as pd
import numpy as np
import json
import pickle
import re
import ast
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

TMDB_PATH = Path('data/tmdb-5000')
WORKING   = Path('data/working_tmdb')
WORKING.mkdir(parents=True, exist_ok=True)

print('Imports OK')
print(f'TMDB path   : {TMDB_PATH}')
print(f'Working dir : {WORKING}')
print(f'Files in TMDB_PATH:')
for p in sorted(TMDB_PATH.iterdir()):
    print(f'  {p.name}')


Imports OK
TMDB path   : data\tmdb-5000
Working dir : data\working_tmdb
Files in TMDB_PATH:
  tmdb_5000_credits.csv
  tmdb_5000_movies.csv


In [3]:
# Load & Merge TMDB Files
print('LOADING TMDB DATA')
print('=' * 60)

print('\n[1/2] Loading tmdb_5000_movies.csv ...')
movies_df = pd.read_csv(TMDB_PATH / 'tmdb_5000_movies.csv')
print(f'  Shape   : {movies_df.shape}')
print(f'  Columns : {list(movies_df.columns)}')

print('\n[2/2] Loading tmdb_5000_credits.csv ...')
credits_df = pd.read_csv(TMDB_PATH / 'tmdb_5000_credits.csv')
print(f'  Shape   : {credits_df.shape}')
print(f'  Columns : {list(credits_df.columns)}')

# Merge on movie_id / id
credits_df.rename(columns={'movie_id': 'id'}, inplace=True)
df = movies_df.merge(credits_df[['id', 'cast', 'crew']], on='id', how='left')
print(f'\n  Merged shape : {df.shape}')
print(f'  Sample titles: {df["title"].head(5).tolist()}')


LOADING TMDB DATA

[1/2] Loading tmdb_5000_movies.csv ...
  Shape   : (4803, 20)
  Columns : ['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']

[2/2] Loading tmdb_5000_credits.csv ...
  Shape   : (4803, 4)
  Columns : ['movie_id', 'title', 'cast', 'crew']

  Merged shape : (4803, 22)
  Sample titles: ['Avatar', "Pirates of the Caribbean: At World's End", 'Spectre', 'The Dark Knight Rises', 'John Carter']


In [4]:
# Parse JSON Fields 
print('=' * 60)
print('PARSING JSON FIELDS')
print('=' * 60)

def safe_parse(val):
    """Safely parse a JSON string or return empty list."""
    if pd.isna(val):
        return []
    try:
        return ast.literal_eval(val)
    except Exception:
        return []

def extract_names(val, key='name', limit=None):
    items = safe_parse(val)
    names = [item[key] for item in items if key in item]
    return names[:limit] if limit else names

def extract_director(crew_val):
    crew = safe_parse(crew_val)
    for member in crew:
        if member.get('job') == 'Director':
            return member.get('name', '')
    return ''

print('Parsing genres, keywords, cast, crew ...')
df['genres_list']   = df['genres'].apply(lambda x: extract_names(x))
df['keywords_list'] = df['keywords'].apply(lambda x: extract_names(x))
df['cast_list']     = df['cast'].apply(lambda x: extract_names(x, limit=5))
df['director']      = df['crew'].apply(extract_director)

# Parse release year
df['year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year

# Clean genre string for display
df['genres_str'] = df['genres_list'].apply(lambda g: '|'.join(g) if g else 'Unknown')

print(f'  Sample genres   : {df["genres_list"].iloc[0]}')
print(f'  Sample keywords : {df["keywords_list"].iloc[0][:5]}')
print(f'  Sample cast     : {df["cast_list"].iloc[0]}')
print(f'  Sample director : {df["director"].iloc[0]}')
print(f'  Year range      : {int(df["year"].min())} – {int(df["year"].dropna().max())}')
print(f'  Total movies    : {len(df):,}')


PARSING JSON FIELDS
Parsing genres, keywords, cast, crew ...
  Sample genres   : ['Action', 'Adventure', 'Fantasy', 'Science Fiction']
  Sample keywords : ['culture clash', 'future', 'space war', 'space colony', 'society']
  Sample cast     : ['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver', 'Stephen Lang', 'Michelle Rodriguez']
  Sample director : James Cameron
  Year range      : 1916 – 2017
  Total movies    : 4,803


In [5]:
# Build Text Descriptions for SBERT 
# overview (plot synopsis)  + keywords + genres + director + top cast
# SBERT will encode this into a 384-dim dense vector.

print('=' * 60)
print(' BUILDING TEXT DESCRIPTIONS')
print('=' * 60)

def build_description(row):
    parts = []

    # Overview is the most important — the actual plot description
    overview = str(row.get('overview', '') or '').strip()
    if overview and overview != 'nan':
        parts.append(overview)

    # Keywords add thematic signal (e.g. "time travel", "based on novel")
    kws = row['keywords_list']
    if kws:
        parts.append('Keywords: ' + ', '.join(kws[:15]))

    # Genres
    genres = row['genres_list']
    if genres:
        parts.append('Genres: ' + ', '.join(genres))

    # Director is highly predictive of style
    director = str(row.get('director', '') or '').strip()
    if director:
        parts.append(f'Director: {director}')

    # Top cast
    cast = row['cast_list']
    if cast:
        parts.append('Cast: ' + ', '.join(cast))

    return ' '.join(parts).strip()

df['description'] = df.apply(build_description, axis=1)

# Fallback: use title if description is empty
df['description'] = df.apply(
    lambda r: r['title'] if not r['description'] else r['description'], axis=1
)

empty = (df['description'].str.strip() == '').sum()
print(f'  Movies with non-empty description : {len(df) - empty:,}')
print(f'  Movies with empty description     : {empty:,}')
print(f'\n  Sample description (first movie):')
print(f'  {df["description"].iloc[0][:300]}')


 BUILDING TEXT DESCRIPTIONS
  Movies with non-empty description : 4,803
  Movies with empty description     : 0

  Sample description (first movie):
  In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Keywords: culture clash, future, space war, space colony, society, space travel, futuristic, romance, space, alien, tribe, a


In [12]:
# Encode with SBERT 
# Encode all movie descriptions using sentence-transformers.
# all-MiniLM-L6-v2 produces 384-dim vectors

print('=' * 60)
print('SBERT ENCODING  ')
print('=' * 60)

from sentence_transformers import SentenceTransformer
print('SentenceTransformer loaded.')
import numpy as np

print('Loading all-MiniLM-L6-v2 ...')
sbert = SentenceTransformer('all-MiniLM-L6-v2')

descriptions = df['description'].tolist()
print(f'Encoding {len(descriptions):,} movie descriptions ...')


embeddings = sbert.encode(
    descriptions,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # L2-normalize so dot product = cosine sim
)

print(f'\nEmbedding matrix shape : {embeddings.shape}')
print(f'Embedding dtype        : {embeddings.dtype}')
print(f'L2 norm of first vec   : {np.linalg.norm(embeddings[0]):.4f}  (should be 1.0)')


SBERT ENCODING  
SentenceTransformer loaded.
Loading all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding 4,803 movie descriptions ...


Batches:   0%|          | 0/38 [00:00<?, ?it/s]


Embedding matrix shape : (4803, 384)
Embedding dtype        : float32
L2 norm of first vec   : 1.0000  (should be 1.0)


In [13]:
# Build Movie Metadata Table 
print('=' * 60)
print('BUILDING METADATA TABLE')
print('=' * 60)

# for the Django API
# movie_id= TMDB id 
meta = df[['id', 'title', 'year', 'genres_str', 'overview',
           'vote_average', 'vote_count', 'popularity']].copy()
meta.rename(columns={
    'id'          : 'movie_id',
    'genres_str'  : 'genres',
    'vote_average': 'rating_mean',
    'vote_count'  : 'rating_count',
}, inplace=True)

meta['year']         = meta['year'].fillna(0).astype(int)
meta['rating_count'] = meta['rating_count'].fillna(0).astype(int)
meta['rating_mean']  = meta['rating_mean'].fillna(0.0).round(2)
meta['popularity']   = meta['popularity'].fillna(0.0).round(2)

# movie_id → row_index mapping (row index = embedding row)

meta = meta.reset_index(drop=True)
meta['row_idx'] = meta.index

print(f'  Metadata table shape : {meta.shape}')
print(f'  Columns              : {list(meta.columns)}')
print()
print('  Top 10 by rating_count:')
print(f'  {"Count":>8}  {"Mean":>5}  Title')
print('  ' + '-' * 50)
for _, row in meta.sort_values('rating_count', ascending=False).head(10).iterrows():
    print(f'  {int(row["rating_count"]):>8,}  {row["rating_mean"]:>5.1f}  {row["title"]} ({row["year"]})')


BUILDING METADATA TABLE
  Metadata table shape : (4803, 9)
  Columns              : ['movie_id', 'title', 'year', 'genres', 'overview', 'rating_mean', 'rating_count', 'popularity', 'row_idx']

  Top 10 by rating_count:
     Count   Mean  Title
  --------------------------------------------------
    13,752    8.1  Inception (2010)
    12,002    8.2  The Dark Knight (2008)
    11,800    7.2  Avatar (2009)
    11,776    7.4  The Avengers (2012)
    10,995    7.4  Deadpool (2016)
    10,867    8.1  Interstellar (2014)
    10,099    7.8  Django Unchained (2012)
     9,742    7.9  Guardians of the Galaxy (2014)
     9,455    6.9  The Hunger Games (2012)
     9,427    7.2  Mad Max: Fury Road (2015)


In [14]:
# Sanity Checks 
print('=' * 60)
print('SANITY CHECKS')
print('=' * 60)

import numpy as np

assert len(embeddings) == len(meta), 'Embedding count must equal metadata row count'
print(f'  [OK] Embedding rows match metadata rows : {len(embeddings):,}')

assert not np.isnan(embeddings).any(), 'NaN found in embeddings'
print(f'  [OK] No NaNs in embedding matrix')

# Verify normalization: all norms should be very close to 1.0
norms = np.linalg.norm(embeddings, axis=1)
assert norms.min() > 0.99 and norms.max() < 1.01, f'Embeddings not normalized: {norms.min():.4f} – {norms.max():.4f}'
print(f'  [OK] All vectors L2-normalized  (min={norms.min():.4f}, max={norms.max():.4f})')

# Quick similarity test: The Matrix should be similar to other sci-fi/action
test_title = 'The Matrix'
test_rows  = meta[meta['title'].str.lower() == test_title.lower()]
if not test_rows.empty:
    test_idx    = int(test_rows.iloc[0]['row_idx'])
    test_vec    = embeddings[[test_idx]]                           # [1, 384]
    sims        = (embeddings @ test_vec.T).squeeze()              # [N]
    sims[test_idx] = -1.0
    top5_idxs   = sims.argsort()[::-1][:5]
    print(f'\n  Similarity test — movies most similar to "{test_title}":')
    for idx in top5_idxs:
        row = meta.iloc[idx]
        print(f'    {sims[idx]:.4f}  {row["title"]} ({row["year"]})')
else:
    print(f'  (skipped similarity test — "{test_title}" not found in dataset)')

print()
print('All sanity checks passed.')


SANITY CHECKS
  [OK] Embedding rows match metadata rows : 4,803
  [OK] No NaNs in embedding matrix
  [OK] All vectors L2-normalized  (min=1.0000, max=1.0000)

  Similarity test — movies most similar to "The Matrix":
    0.6816  The Matrix Revolutions (2003)
    0.6608  The Matrix Reloaded (2003)
    0.5604  Commando (1985)
    0.5443  Terminator Genisys (2015)
    0.5419  eXistenZ (1999)

All sanity checks passed.


In [15]:
# Save All Artifacts 
print('=' * 60)
print('STEP 7 — SAVING ARTIFACTS')
print('=' * 60)

def save(name, obj):
    path = WORKING / f'{name}.pkl'
    with open(path, 'wb') as f:
        pickle.dump(obj, f, protocol=4)
    mb = path.stat().st_size / 1e6
    print(f'  {name:<25s}  {mb:6.1f} MB')

save('embeddings',   embeddings)   # np.ndarray [4803, 384] float32 normalized
save('metadata',     meta)         # pd.DataFrame with movie_id, title, etc.

print()
print('Summary:')
print(f'  Total movies    : {len(meta):,}')
print(f'  Embedding shape : {embeddings.shape}')
print(f'  Embedding dim   : {embeddings.shape[1]}')
print()



STEP 7 — SAVING ARTIFACTS
  embeddings                    7.4 MB
  metadata                      1.9 MB

Summary:
  Total movies    : 4,803
  Embedding shape : (4803, 384)
  Embedding dim   : 384

